# NB3 — ScienceWorld Dataset Generation (REBUILT)

**Purpose:** Generate a reproducible ScienceWorld pilot dataset from fresh simulator runs, standardise records, preserve raw observations, and validate the resulting dataset before integration in NB4.

**Important scope:** NB3 separates *observed action effects* from *reversibility labels*. A changed object tree is not automatically irreversible. Records without a completed, defensible bounded-recovery test remain `unlabelled_candidate`. The pilot contains two explicitly tested object-level cases from NB2 (picture relocation and orange consumption) plus a broader catalogue of valid task actions for later controlled expansion.

**Authoritative inputs:** Fresh ScienceWorld 1.2.3 runs only. This notebook does not read legacy `irrev_backup` data or old generated datasets.


## 1. Project setup

Uses the rebuilt project roots:
- Local project: `/content/irrev`
- Main Drive project: `/content/drive/MyDrive/ir`
- Independent backup snapshots: `/content/drive/MyDrive/ir_final`

The notebook creates dedicated NB3 data, result, and report directories. Existing outputs are preserved by writing timestamped run files.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import os, sys, json, hashlib, copy, shutil, time, platform, subprocess, re
import importlib.metadata as metadata

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/irrev")
DRIVE_ROOT = Path("/content/drive/MyDrive/ir")
BACKUP_ROOT = Path("/content/drive/MyDrive/ir_final")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "scienceworld" / "raw"
PROCESSED_DIR = DATA_DIR / "scienceworld" / "processed"
RESULT_DIR = PROJECT_ROOT / "results" / "scienceworld_dataset"
REPORT_DIR = PROJECT_ROOT / "reports"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

for path in (PROJECT_ROOT, DRIVE_ROOT, BACKUP_ROOT, RAW_DIR, PROCESSED_DIR,
             RESULT_DIR, REPORT_DIR, NOTEBOOK_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
os.environ["JAVA_TOOL_OPTIONS"] = "-Xlog:os+container=off"

print("Project:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Python:", platform.python_version())
subprocess.run(["java", "-version"], check=False)


Mounted at /content/drive
Project: /content/irrev
Raw data: /content/irrev/data/scienceworld/raw
Processed data: /content/irrev/data/scienceworld/processed
Python: 3.13.18


CompletedProcess(args=['java', '-version'], returncode=0)

## 2. Install and initialise ScienceWorld

The environment version is pinned to `scienceworld==1.2.3`, matching NB1 and NB2. Every experiment starts from a fresh environment reset.


In [2]:
try:
    import scienceworld
    from scienceworld import ScienceWorldEnv
    print("ScienceWorld already installed:", getattr(scienceworld, "__version__", "version not exposed"))
except ImportError:
    %pip install -q scienceworld==1.2.3
    import scienceworld
    from scienceworld import ScienceWorldEnv

SCIENCEWORLD_VERSION = getattr(scienceworld, "__version__", "version not exposed")

def make_env(task, variation=0, simplification="easy", step_limit=100):
    env = ScienceWorldEnv(envStepLimit=step_limit)
    env.load(task, variation, simplification)
    observation, info = env.reset()
    return env, observation, info

def safe_close(env):
    try:
        env.close()
    except Exception:
        pass

def current_action_list(env):
    try:
        return list(env.get_valid_action_object_combinations())
    except Exception:
        return []

def execute_if_valid(env, action):
    valid = current_action_list(env)
    if action not in valid:
        return {"action": action, "executed": False,
                "reason": "Action not present in current valid-action list",
                "moves": env.get_num_moves()}
    result = env.step(action)
    return {"action": action, "executed": True, "observation": result[0],
            "reward": result[1], "done": result[2],
            "moves": env.get_num_moves()}

print("ScienceWorld version:", SCIENCEWORLD_VERSION)


ScienceWorld version: 1.2.3


## 3. Canonical state and record utilities

Structural fingerprints exclude only `temperatureC`, consistent with NB2. Full fingerprints retain all object-tree fields. These are observable object-tree representations, not complete simulator-state proofs.


In [3]:
def normalise_tree(value, exclude_temperature=False):
    if isinstance(value, dict):
        return {
            key: normalise_tree(child, exclude_temperature)
            for key, child in sorted(value.items())
            if not (exclude_temperature and key == "temperatureC")
        }
    if isinstance(value, list):
        return [normalise_tree(item, exclude_temperature) for item in value]
    return value

def fingerprint(tree, exclude_temperature=False):
    payload = json.dumps(normalise_tree(tree, exclude_temperature),
                         sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def structural_fingerprint(tree):
    return fingerprint(tree, exclude_temperature=True)

def full_state_fingerprint(tree):
    return fingerprint(tree, exclude_temperature=False)

def find_all_objects(tree):
    found = []
    def walk(node, path=""):
        if isinstance(node, dict):
            if "uuid" in node:
                found.append((path, node))
            for key, child in node.items():
                if isinstance(child, (dict, list)):
                    walk(child, f"{path}/{key}")
        elif isinstance(node, list):
            for i, child in enumerate(node):
                walk(child, f"{path}[{i}]")
    walk(tree)
    return found

def object_summaries(tree):
    return [
        {"path": path, "uuid": str(obj.get("uuid")),
         "name": obj.get("name"), "isDeleted": obj.get("isDeleted"),
         "object": copy.deepcopy(obj)}
        for path, obj in find_all_objects(tree)
    ]

def stable_id(*parts):
    raw = "||".join(str(x) for x in parts)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]

def write_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    return path

def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

print("Canonical utilities defined.")


Canonical utilities defined.


## 4. Pre-register the ScienceWorld pilot scenarios

The two controlled scenarios below are grounded in NB2 observations:
1. `picture_relocation`: move the same picture from the hallway to the art studio and return it to the hallway within three recovery actions.
2. `orange_consumption`: consume the orange identified in the agent's inventory and observe whether that specific object UUID remains in the object tree.

The second case is recorded as an *object-level consumption candidate*. A disappearance is not treated as absolute proof of irreversibility, and the presence of another orange is not confused with recovery of the consumed object.


In [4]:
PILOT_SCENARIOS = [
    {
        "scenario_name": "picture_relocation",
        "task": "boil",
        "variation": 0,
        "initialisation_action": "look at orange",
        "intervention_actions": [
            "pick up picture", "go to art studio", "put down picture"
        ],
        "recovery_actions": [
            "pick up picture", "go to hallway", "put down picture"
        ],
        "object_name": "picture",
        "expected_interpretation": "bounded_structural_recovery_candidate",
        "label": "reversible",
        "label_scope": "object-level; structural observable recovery within 3 actions",
        "label_confidence": "pilot_observed"
    },
    {
        "scenario_name": "orange_consumption",
        "task": "boil",
        "variation": 0,
        "initialisation_action": "look at orange",
        "intervention_actions": ["eat orange"],
        "recovery_actions": [],
        "object_name": "orange",
        "expected_interpretation": "object_level_consumption_candidate",
        "label": "unlabelled_candidate",
        "label_scope": "specific target UUID disappearance; recovery not yet searched",
        "label_confidence": "not_assigned"
    }
]

print(json.dumps(PILOT_SCENARIOS, indent=2))


[
  {
    "scenario_name": "picture_relocation",
    "task": "boil",
    "variation": 0,
    "initialisation_action": "look at orange",
    "intervention_actions": [
      "pick up picture",
      "go to art studio",
      "put down picture"
    ],
    "recovery_actions": [
      "pick up picture",
      "go to hallway",
      "put down picture"
    ],
    "object_name": "picture",
    "expected_interpretation": "bounded_structural_recovery_candidate",
    "label": "reversible",
    "label_scope": "object-level; structural observable recovery within 3 actions",
    "label_confidence": "pilot_observed"
  },
  {
    "scenario_name": "orange_consumption",
    "task": "boil",
    "variation": 0,
    "initialisation_action": "look at orange",
    "intervention_actions": [
      "eat orange"
    ],
    "recovery_actions": [],
    "object_name": "orange",
    "expected_interpretation": "object_level_consumption_candidate",
    "label": "unlabelled_candidate",
    "label_scope": "specific targ

## 5. Execute the controlled pilot scenarios

Each scenario uses a fresh `boil` environment and the same initialisation action. The notebook records the target UUIDs before and after intervention/recovery, action validity, observations, move counts, and structural/full-state fingerprints.

The picture scenario receives a bounded structural-recovery label only if all specified actions execute and the H3 structural fingerprint equals the post-initialisation baseline. The orange scenario remains unlabelled unless the target-object recovery question is independently tested.


In [5]:
def objects_named(tree, name):
    return [
        {"path": path, "uuid": str(obj.get("uuid")),
         "name": obj.get("name"), "isDeleted": obj.get("isDeleted"),
         "object": copy.deepcopy(obj)}
        for path, obj in find_all_objects(tree)
        if str(obj.get("name", "")).lower() == str(name).lower()
    ]

def run_pilot_scenario(spec):
    env, obs, info = make_env(spec["task"], spec["variation"])
    result = {
        "scenario_name": spec["scenario_name"],
        "task": spec["task"],
        "variation": spec["variation"],
        "initialisation_action": None,
        "baseline": {},
        "intervention_steps": [],
        "recovery_steps": [],
        "target_objects_before": [],
        "target_objects_after": [],
        "label": spec["label"],
        "label_scope": spec["label_scope"],
        "label_confidence": spec["label_confidence"]
    }
    try:
        init_record = execute_if_valid(env, spec["initialisation_action"])
        result["initialisation_action"] = init_record
        if not init_record["executed"]:
            result["status"] = "initialisation_failed"
            return result

        baseline_tree = copy.deepcopy(env.getObjectTree())
        result["baseline"] = {
            "moves": env.get_num_moves(),
            "structural_fingerprint": structural_fingerprint(baseline_tree),
            "full_state_fingerprint": full_state_fingerprint(baseline_tree)
        }
        result["target_objects_before"] = objects_named(baseline_tree, spec["object_name"])

        intervention_ok = True
        for action in spec["intervention_actions"]:
            rec = execute_if_valid(env, action)
            result["intervention_steps"].append(rec)
            if not rec["executed"] or rec.get("done"):
                intervention_ok = False
                break

        if intervention_ok:
            intervention_tree = copy.deepcopy(env.getObjectTree())
            result["after_intervention"] = {
                "moves": env.get_num_moves(),
                "structural_fingerprint": structural_fingerprint(intervention_tree),
                "full_state_fingerprint": full_state_fingerprint(intervention_tree),
                "target_objects": objects_named(intervention_tree, spec["object_name"])
            }

            for action in spec["recovery_actions"]:
                rec = execute_if_valid(env, action)
                result["recovery_steps"].append(rec)
                if not rec["executed"] or rec.get("done"):
                    break

        final_tree = copy.deepcopy(env.getObjectTree())
        result["target_objects_after"] = objects_named(final_tree, spec["object_name"])
        result["final"] = {
            "moves": env.get_num_moves(),
            "structural_fingerprint": structural_fingerprint(final_tree),
            "full_state_fingerprint": full_state_fingerprint(final_tree)
        }

        all_actions_executed = (
            intervention_ok
            and all(x["executed"] for x in result["intervention_steps"])
            and all(x["executed"] for x in result["recovery_steps"])
            and len(result["recovery_steps"]) == len(spec["recovery_actions"])
        )
        result["all_planned_actions_executed"] = all_actions_executed
        result["structural_recovered"] = (
            structural_fingerprint(final_tree) == result["baseline"]["structural_fingerprint"]
        )
        result["full_state_recovered"] = (
            full_state_fingerprint(final_tree) == result["baseline"]["full_state_fingerprint"]
        )

        if spec["scenario_name"] == "picture_relocation":
            if all_actions_executed and result["structural_recovered"]:
                result["label"] = "reversible"
                result["label_scope"] = "object-level; structural observable recovery within 3 actions"
                result["label_confidence"] = "pilot_observed"
            else:
                result["label"] = "unlabelled_candidate"
                result["label_scope"] = "bounded recovery not demonstrated"
                result["label_confidence"] = "not_assigned"

        elif spec["scenario_name"] == "orange_consumption":
            before_ids = {x["uuid"] for x in result["target_objects_before"]}
            after_ids = {x["uuid"] for x in result["target_objects_after"]}
            result["target_uuids_missing_after"] = sorted(before_ids - after_ids)
            result["label"] = "unlabelled_candidate"
            result["label_scope"] = "target UUID disappearance observed; bounded recovery not tested"
            result["label_confidence"] = "not_assigned"

        result["status"] = "completed"
        return result
    finally:
        safe_close(env)

pilot_results = [run_pilot_scenario(spec) for spec in PILOT_SCENARIOS]
print(json.dumps(pilot_results, indent=2, ensure_ascii=False, default=str))


[
  {
    "scenario_name": "picture_relocation",
    "task": "boil",
    "variation": 0,
    "initialisation_action": {
      "action": "look at orange",
      "executed": true,
      "observation": "an orange",
      "reward": 0,
      "done": false,
      "moves": 1
    },
    "baseline": {
      "moves": 1,
      "structural_fingerprint": "ce0025a1f8e01d2e931d56d65a527d4419ef3834e5525e581d5a5a616826d3fb",
      "full_state_fingerprint": "84b10c581802000082afd4fc5b826ec56a381e2304e9962e7bba1500d68c1269"
    },
    "intervention_steps": [
      {
        "action": "pick up picture",
        "executed": true,
        "observation": "You move the picture to the inventory.",
        "reward": 0,
        "done": false,
        "moves": 2
      },
      {
        "action": "go to art studio",
        "executed": true,
        "observation": "You move to the art studio.",
        "reward": 0,
        "done": false,
        "moves": 3
      },
      {
        "action": "put down picture",
  

## 6. Build a broader valid-action catalogue

This is a discovery pass, not a reversibility labelling pass. For each selected task, it records the task's valid action strings from a fresh reset, plus the initial object-tree fingerprints and object summaries. No action is executed in this section.

The catalogue helps design later controlled interventions and recovery sequences without assuming that arbitrary valid actions are irreversible.


In [6]:
TASKS_FOR_CATALOGUE = [
    "boil", "chemistry-mix", "freeze", "melt", "cook",
    "grow-plant", "clean-something", "find-living-thing",
    "identify-life-stages-1", "measure-melting-point"
]

action_catalogue = []
catalogue_errors = []

for task in TASKS_FOR_CATALOGUE:
    env = None
    try:
        env, obs, info = make_env(task, variation=0)
        tree = copy.deepcopy(env.getObjectTree())
        actions = current_action_list(env)
        action_catalogue.append({
            "task": task,
            "variation": 0,
            "valid_action_count": len(actions),
            "valid_actions": actions,
            "initial_structural_fingerprint": structural_fingerprint(tree),
            "initial_full_state_fingerprint": full_state_fingerprint(tree),
            "initial_object_count": len(find_all_objects(tree)),
            "initial_objects": [
                {"uuid": str(obj.get("uuid")), "name": obj.get("name"),
                 "path": path, "isDeleted": obj.get("isDeleted")}
                for path, obj in find_all_objects(tree)
            ]
        })
    except Exception as exc:
        catalogue_errors.append({"task": task, "error": repr(exc)})
    finally:
        if env is not None:
            safe_close(env)

print("Tasks catalogued:", len(action_catalogue))
print("Catalogue errors:", json.dumps(catalogue_errors, indent=2))
print("Valid actions:", sum(x["valid_action_count"] for x in action_catalogue))


Tasks catalogued: 7
Catalogue errors: [
  {
    "task": "cook",
    "error": "ValueError(\"Unknown taskName: 'cook'. Supported tasks are: ['boil', 'change-the-state-of-matter-of', 'chemistry-mix', 'chemistry-mix-paint-secondary-color', 'chemistry-mix-paint-tertiary-color', 'find-animal', 'find-living-thing', 'find-non-living-thing', 'find-plant', 'freeze', 'grow-fruit', 'grow-plant', 'identify-life-stages-1', 'identify-life-stages-2', 'inclined-plane-determine-angle', 'inclined-plane-friction-named-surfaces', 'inclined-plane-friction-unnamed-surfaces', 'lifespan-longest-lived', 'lifespan-longest-lived-then-shortest-lived', 'lifespan-shortest-lived', 'measure-melting-point-known-substance', 'measure-melting-point-unknown-substance', 'melt', 'mendelian-genetics-known-plant', 'mendelian-genetics-unknown-plant', 'power-component', 'power-component-renewable-vs-nonrenewable-energy', 'test-conductivity', 'test-conductivity-of-unknown-substances', 'use-thermometer']\")"
  },
  {
    "task": "

## 7. Create canonical dataset records

Schema fields are stable across environments and will be used by NB4 to integrate ScienceWorld with the Python Sandbox dataset.

`label` is one of:
- `reversible`
- `partially_reversible`
- `irreversible`
- `unlabelled_candidate`

`unlabelled_candidate` is a data-collection status, not a fourth scientific class. It must not be silently converted into one of the three target labels.


In [7]:
def canonical_record(scenario, result):
    record_id = stable_id("scienceworld", scenario["scenario_name"],
                          scenario["task"], scenario["variation"])
    return {
        "record_id": record_id,
        "pair_id": None,
        "environment": "scienceworld",
        "scenario_name": scenario["scenario_name"],
        "task": scenario["task"],
        "variation": scenario["variation"],
        "action_sequence": scenario["intervention_actions"],
        "recovery_sequence": scenario["recovery_actions"],
        "target_object": scenario["object_name"],
        "label": result.get("label", "unlabelled_candidate"),
        "label_scope": result.get("label_scope"),
        "label_confidence": result.get("label_confidence"),
        "horizon": len(scenario["recovery_actions"]),
        "initialisation_action": scenario["initialisation_action"],
        "initial_structural_fingerprint": result.get("baseline", {}).get("structural_fingerprint"),
        "final_structural_fingerprint": result.get("final", {}).get("structural_fingerprint"),
        "structural_recovered": result.get("structural_recovered"),
        "full_state_recovered": result.get("full_state_recovered"),
        "all_planned_actions_executed": result.get("all_planned_actions_executed"),
        "target_objects_before": result.get("target_objects_before", []),
        "target_objects_after": result.get("target_objects_after", []),
        "target_uuids_missing_after": result.get("target_uuids_missing_after", []),
        "status": result.get("status"),
        "provenance": {
            "notebook": "03_scienceworld_dataset",
            "scienceworld_version": SCIENCEWORLD_VERSION,
            "generation_method": "fresh_environment_controlled_pilot",
            "created_utc": datetime.now(timezone.utc).isoformat()
        }
    }

scenario_by_name = {x["scenario_name"]: x for x in PILOT_SCENARIOS}
canonical_records = [
    canonical_record(scenario_by_name[result["scenario_name"]], result)
    for result in pilot_results
]

print("Canonical pilot records:", len(canonical_records))
print("Label counts:", dict(Counter(x["label"] for x in canonical_records)))


Canonical pilot records: 2
Label counts: {'reversible': 1, 'unlabelled_candidate': 1}


## 8. Dataset quality checks

The checks below validate schema completeness, unique record IDs, valid label vocabulary, provenance, and internal consistency. They deliberately do not require every record to have a scientific label: unresolved candidates must remain explicitly unresolved.


In [8]:
ALLOWED_LABELS = {
    "reversible", "partially_reversible", "irreversible", "unlabelled_candidate"
}
REQUIRED_FIELDS = [
    "record_id", "environment", "scenario_name", "task", "variation",
    "action_sequence", "recovery_sequence", "target_object", "label",
    "label_scope", "label_confidence", "provenance"
]

validation_errors = []
record_ids = [x.get("record_id") for x in canonical_records]

if len(record_ids) != len(set(record_ids)):
    validation_errors.append("Duplicate record_id values detected.")

for i, record in enumerate(canonical_records):
    missing = [key for key in REQUIRED_FIELDS if key not in record]
    if missing:
        validation_errors.append(f"Record {i} missing fields: {missing}")
    if record.get("label") not in ALLOWED_LABELS:
        validation_errors.append(f"Record {i} has invalid label: {record.get('label')}")
    if record.get("environment") != "scienceworld":
        validation_errors.append(f"Record {i} has unexpected environment.")
    if not record.get("provenance", {}).get("scienceworld_version"):
        validation_errors.append(f"Record {i} missing ScienceWorld version provenance.")
    if record.get("label") == "reversible" and record.get("structural_recovered") is not True:
        validation_errors.append(f"Record {i} labelled reversible without structural recovery.")

validation_summary = {
    "record_count": len(canonical_records),
    "unique_record_ids": len(set(record_ids)),
    "label_distribution": dict(Counter(x["label"] for x in canonical_records)),
    "validation_error_count": len(validation_errors),
    "validation_errors": validation_errors,
    "passed": len(validation_errors) == 0
}
print(json.dumps(validation_summary, indent=2))
if validation_errors:
    raise RuntimeError("NB3 dataset validation failed; resolve errors before proceeding.")


{
  "record_count": 2,
  "unique_record_ids": 2,
  "label_distribution": {
    "reversible": 1,
    "unlabelled_candidate": 1
  },
  "validation_error_count": 0,
  "validation_errors": [],
  "passed": true
}


## 9. Save raw, processed, and report artefacts

Raw pilot results preserve the detailed simulator traces. The processed JSONL contains canonical records. The action catalogue is kept separately because it is a discovery artefact, not labelled training data.


In [9]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

raw_pilot_path = RAW_DIR / f"scienceworld_pilot_raw_{run_timestamp}.json"
raw_catalogue_path = RAW_DIR / f"scienceworld_action_catalogue_{run_timestamp}.json"
processed_path = PROCESSED_DIR / f"scienceworld_pilot_canonical_{run_timestamp}.jsonl"
validation_path = RESULT_DIR / f"NB03_validation_{run_timestamp}.json"
report_path = REPORT_DIR / f"NB03_scienceworld_dataset_report_{run_timestamp}.json"

raw_pilot_path.write_text(json.dumps(pilot_results, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
raw_catalogue_path.write_text(json.dumps(action_catalogue, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
write_jsonl(processed_path, canonical_records)
validation_path.write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")

nb3_report = {
    "notebook": "03_scienceworld_dataset",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "scienceworld_version": SCIENCEWORLD_VERSION,
    "run_timestamp": run_timestamp,
    "pilot_scenario_count": len(pilot_results),
    "canonical_record_count": len(canonical_records),
    "label_distribution": dict(Counter(x["label"] for x in canonical_records)),
    "catalogue_task_count": len(action_catalogue),
    "catalogue_valid_action_count": sum(x["valid_action_count"] for x in action_catalogue),
    "catalogue_errors": catalogue_errors,
    "validation": validation_summary,
    "artefacts": {
        "raw_pilot": str(raw_pilot_path),
        "raw_action_catalogue": str(raw_catalogue_path),
        "processed_jsonl": str(processed_path),
        "validation": str(validation_path)
    },
    "methodological_notes": [
        "Observed state changes are not automatically labelled irreversible.",
        "Unresolved examples remain unlabelled_candidate.",
        "The picture relocation label is bounded object-level structural recovery within three actions.",
        "Orange consumption is recorded as a target-object disappearance candidate; recovery has not been searched.",
        "Object-tree fingerprints are observable-state measures, not complete simulator-state proofs."
    ]
}
report_path.write_text(json.dumps(nb3_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("Saved raw pilot:", raw_pilot_path)
print("Saved action catalogue:", raw_catalogue_path)
print("Saved canonical JSONL:", processed_path)
print("Saved validation:", validation_path)
print("Saved report:", report_path)


Saved raw pilot: /content/irrev/data/scienceworld/raw/scienceworld_pilot_raw_20261123T123703338645Z.json
Saved action catalogue: /content/irrev/data/scienceworld/raw/scienceworld_action_catalogue_20261123T123703338645Z.json
Saved canonical JSONL: /content/irrev/data/scienceworld/processed/scienceworld_pilot_canonical_20261123T123703338645Z.jsonl
Saved validation: /content/irrev/results/scienceworld_dataset/NB03_validation_20261123T123703338645Z.json
Saved report: /content/irrev/reports/NB03_scienceworld_dataset_report_20261123T123703338645Z.json


## 10. Reload and verify saved artefacts

This checks that the on-disk JSONL can be reloaded and preserves record IDs and labels.


In [10]:
reloaded_records = read_jsonl(processed_path)
assert len(reloaded_records) == len(canonical_records)
assert [x["record_id"] for x in reloaded_records] == [x["record_id"] for x in canonical_records]
assert [x["label"] for x in reloaded_records] == [x["label"] for x in canonical_records]

for path in (raw_pilot_path, raw_catalogue_path, processed_path, validation_path, report_path):
    assert path.exists() and path.stat().st_size > 0, f"Missing/empty artefact: {path}"

print("NB3 artefact reload verification passed.")
for path in (raw_pilot_path, raw_catalogue_path, processed_path, validation_path, report_path):
    print(path.name, "|", path.stat().st_size, "bytes")


NB3 artefact reload verification passed.
scienceworld_pilot_raw_20261123T123703338645Z.json | 21279 bytes
scienceworld_action_catalogue_20261123T123703338645Z.json | 509743 bytes
scienceworld_pilot_canonical_20261123T123703338645Z.jsonl | 9192 bytes
NB03_validation_20261123T123703338645Z.json | 207 bytes
NB03_scienceworld_dataset_report_20261123T123703338645Z.json | 4538 bytes


## 11. Timestamped Drive backup with SHA-256 verification

Run after the experiment, report, and validation cells. Existing snapshots are preserved; this cell refuses to overwrite a snapshot.


In [11]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

snapshot_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
snapshot_name = f"NB3_snapshot_{snapshot_timestamp}"

source_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file() and "__pycache__" not in p.parts
    and ".ipynb_checkpoints" not in p.parts
]
if not source_files:
    raise RuntimeError("No project files found to back up.")

backup_results = []
for root in (DRIVE_ROOT, BACKUP_ROOT):
    destination = root / snapshot_name
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite existing snapshot: {destination}")
    destination.mkdir(parents=True)
    failures = []
    for src in source_files:
        rel = src.relative_to(PROJECT_ROOT)
        dst = destination / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        if src.stat().st_size != dst.stat().st_size or sha256_file(src) != sha256_file(dst):
            failures.append(str(rel))
    backup_results.append({
        "destination": str(destination),
        "file_count": len(source_files),
        "verification_failures": failures
    })

if any(x["verification_failures"] for x in backup_results):
    raise RuntimeError(f"Backup verification failed: {backup_results}")

manifest = {
    "notebook": "03_scienceworld_dataset",
    "snapshot_name": snapshot_name,
    "timestamp_utc": snapshot_timestamp,
    "source_root": str(PROJECT_ROOT),
    "file_count": len(source_files),
    "files": [
        {"path": str(p.relative_to(PROJECT_ROOT)),
         "size_bytes": p.stat().st_size,
         "sha256": sha256_file(p)}
        for p in sorted(source_files)
    ],
    "backup_results": backup_results
}
manifest_path = REPORT_DIR / f"{snapshot_name}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

for item in backup_results:
    dst_manifest = Path(item["destination"]) / manifest_path.name
    shutil.copy2(manifest_path, dst_manifest)
    if sha256_file(manifest_path) != sha256_file(dst_manifest):
        raise RuntimeError(f"Manifest verification failed: {dst_manifest}")

print("NB3 BACKUP VERIFIED")
print("Snapshot:", snapshot_name)
for item in backup_results:
    print(item["destination"])
    print("Files verified:", item["file_count"])
    print("Failures:", item["verification_failures"])
print("Manifest:", manifest_path)


NB3 BACKUP VERIFIED
Snapshot: NB3_snapshot_20261123T123703377799Z
/content/drive/MyDrive/ir/NB3_snapshot_20261123T123703377799Z
Files verified: 5
Failures: []
/content/drive/MyDrive/ir_final/NB3_snapshot_20261123T123703377799Z
Files verified: 5
Failures: []
Manifest: /content/irrev/reports/NB3_snapshot_20261123T123703377799Z_manifest.json


## 12. NB3 completion checklist

NB3 is ready to close when:
- all pilot actions and any invalid/unavailable actions are explicitly recorded;
- the picture scenario's H3 structural-recovery result is consistent with NB2;
- the orange consumption record remains unlabelled unless target-object recovery is actually tested;
- the action catalogue is saved separately from labelled data;
- canonical JSONL passes validation and reload checks;
- both timestamped Drive snapshots pass SHA-256 verification;
- the executed notebook is saved in the project `notebooks` folder or otherwise preserved.

**Expected scope:** This is a controlled pilot and task/action discovery pass, not yet the full 2,000-record integrated dataset. NB4 integrates the ScienceWorld records with the separately generated Python Sandbox records and standardises the combined schema.
